# 04 OHLCV 1m Closeout

Notebook de cierre ejecutivo para `ohlcv_1m` tras el refino de `schema` y `vw`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 140)

ROOT = Path(r"C:/TSIS_Data/02_backtest_SmallCaps")
RUN_MATERIALIZED = ROOT / 'runs/backtest/ohlcv_1m_v2_materialized/ohlcv_1m_current_full'
RUN_VALIDATION = ROOT / 'runs/backtest/ohlcv_1m_v2_validation/ohlcv_1m_validate_full'
AUDIT_OUT = RUN_MATERIALIZED / 'root_cause_operational_outputs'
CURRENT_PATH = RUN_MATERIALIZED / 'ohlcv_1m_current.parquet'

ARTIFACTS = {
    'rescue_schema_only': AUDIT_OUT / 'rescue_schema_only.parquet',
    'rescue_schema_plus_vw': AUDIT_OUT / 'rescue_schema_plus_vw.parquet',
    'hard_quarantine': AUDIT_OUT / 'hard_quarantine.parquet',
    'operational_decision_summary': AUDIT_OUT / 'operational_decision_summary.parquet',
    'price_invalid_resolution': AUDIT_OUT / 'price_invalid_resolution.parquet',
    'price_invalid_resolution_summary': AUDIT_OUT / 'price_invalid_resolution_summary.parquet',
}

materialization_summary = json.loads((RUN_MATERIALIZED / 'materialization_summary.json').read_text(encoding='utf-8'))
validation_manifest = json.loads((RUN_VALIDATION / 'validation_run_manifest.json').read_text(encoding='utf-8'))

def load_artifact(name: str) -> pd.DataFrame:
    return pd.read_parquet(ARTIFACTS[name])

def ensure_vw_refined(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ['m.vw_outside_range_rows', 'm.rows_after_parse', 'm.active_days', 'm.max_gap_days', 'm.vw_mean', 'm.v_sum', 'm.n_sum']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    out['vw_ratio_pct'] = np.where(out['m.rows_after_parse'] > 0, out['m.vw_outside_range_rows'] / out['m.rows_after_parse'] * 100.0, np.nan)
    out['vw_per_active_day'] = np.where(out['m.active_days'] > 0, out['m.vw_outside_range_rows'] / out['m.active_days'], np.nan)
    out['vw_taxonomy'] = np.select(
        [
            out['vw_ratio_pct'].le(1),
            out['vw_ratio_pct'].gt(1) & out['vw_ratio_pct'].le(5),
            out['vw_ratio_pct'].gt(5) & out['m.rows_after_parse'].lt(100),
            out['vw_ratio_pct'].gt(5) & out['m.rows_after_parse'].ge(100) & out['m.vw_outside_range_rows'].lt(100),
            out['vw_ratio_pct'].gt(5) & out['m.rows_after_parse'].ge(100) & out['m.vw_outside_range_rows'].ge(100) & out['vw_per_active_day'].lt(25),
            out['vw_ratio_pct'].gt(5) & out['m.rows_after_parse'].ge(100) & out['m.vw_outside_range_rows'].ge(100) & out['vw_per_active_day'].ge(25),
        ],
        [
            'vw_mild_low_ratio',
            'vw_moderate_ratio',
            'vw_severe_tiny_base',
            'vw_severe_small_mass',
            'vw_severe_large_mass_diffuse',
            'vw_severe_large_mass_persistent',
        ],
        default='vw_other',
    )
    out['quality_policy'] = np.select(
        [
            out.get('operational_decision', pd.Series('', index=out.index)).eq('RESCUE_SCHEMA_ONLY'),
            out['vw_taxonomy'].eq('vw_mild_low_ratio'),
            out['vw_taxonomy'].isin(['vw_moderate_ratio', 'vw_severe_tiny_base', 'vw_severe_small_mass']),
            out['vw_taxonomy'].isin(['vw_severe_large_mass_diffuse', 'vw_severe_large_mass_persistent']),
            out.get('operational_decision', pd.Series('', index=out.index)).isin(['QUARANTINE_PARSE_INVALID', 'QUARANTINE_PRICE_INVALID']),
        ],
        ['good', 'good', 'review', 'bad', 'bad'],
        default='review',
    )
    return out

## Preflight

Verifica que los artefactos operativos necesarios existen.

In [ ]:
missing = [k for k, p in ARTIFACTS.items() if not p.exists()]
if missing:
    raise RuntimeError({'missing_artifacts': missing})
display(pd.DataFrame([{'artifact': k, 'path': str(p), 'exists': p.exists(), 'size_bytes': int(p.stat().st_size)} for k, p in ARTIFACTS.items()]))

## Alcance Auditado

In [ ]:
selected_files = int(sum(batch.get('files_selected', batch.get('rows', 0)) for batch in validation_manifest.get('batch_files', [])))
finished_at_utc = materialization_summary.get('materialized_at_utc') or validation_manifest.get('updated_utc')
scope_df = pd.DataFrame([{
    'current_path': str(CURRENT_PATH),
    'events_rows': materialization_summary['events_rows'],
    'current_rows': materialization_summary['current_rows'],
    'validation_outdir': materialization_summary['validation_outdir'],
    'selected_files': selected_files,
    'workers': validation_manifest['workers'],
    'finished_at_utc': finished_at_utc,
}])
display(scope_df)

## Decision Operativa Base

In [ ]:
decision_summary = load_artifact('operational_decision_summary')
display(decision_summary)
plt.figure(figsize=(9,4))
sns.barplot(data=decision_summary, x='operational_decision', y='files', hue='operational_decision', dodge=False, legend=False)
plt.xticks(rotation=20)
plt.title('Base operational decision')
plt.tight_layout()

## Taxonom?a Refinada De VW

In [ ]:
schema_plus_vw = ensure_vw_refined(load_artifact('rescue_schema_plus_vw'))
vw_summary = schema_plus_vw.groupby('vw_taxonomy').agg(files=('file','count'), tickers=('ticker','nunique'), median_rows=('m.rows_after_parse','median'), median_vw_rows=('m.vw_outside_range_rows','median'), median_ratio=('vw_ratio_pct','median'), median_per_day=('vw_per_active_day','median')).reset_index().sort_values('files', ascending=False)
vw_summary['pct_of_schema_plus_vw'] = 100.0 * vw_summary['files'] / max(len(schema_plus_vw), 1)
display(vw_summary)
plt.figure(figsize=(12,4))
sns.barplot(data=vw_summary, x='vw_taxonomy', y='files', hue='vw_taxonomy', dodge=False, legend=False)
plt.xticks(rotation=20, ha='right')
plt.title('Refined VW taxonomy')
plt.tight_layout()

## Pol?tica Good Review Bad

In [ ]:
schema_only = load_artifact('rescue_schema_only').copy()
schema_only['quality_policy'] = 'good'
schema_plus_vw = ensure_vw_refined(load_artifact('rescue_schema_plus_vw'))
hard_q = load_artifact('hard_quarantine').copy()
hard_q['quality_policy'] = np.where(hard_q['operational_decision'].isin(['QUARANTINE_PARSE_INVALID','QUARANTINE_PRICE_INVALID']), 'bad', 'review')
policy_rows = [
    {'bucket': 'RESCUE_SCHEMA_ONLY', 'policy': 'good', 'reason': 'schema issue only'},
    {'bucket': 'vw_mild_low_ratio', 'policy': 'good', 'reason': 'low-ratio vw noise'},
    {'bucket': 'vw_moderate_ratio', 'policy': 'review', 'reason': 'moderate vw deviation'},
    {'bucket': 'vw_severe_tiny_base', 'policy': 'review', 'reason': 'small base inflates ratio'},
    {'bucket': 'vw_severe_small_mass', 'policy': 'review', 'reason': 'severe but still small mass'},
    {'bucket': 'vw_severe_large_mass_diffuse', 'policy': 'bad', 'reason': 'large affected mass'},
    {'bucket': 'vw_severe_large_mass_persistent', 'policy': 'bad', 'reason': 'large mass and persistent by day'},
    {'bucket': 'QUARANTINE_PARSE_INVALID', 'policy': 'bad', 'reason': 'parse invalid'},
    {'bucket': 'QUARANTINE_PRICE_INVALID', 'policy': 'bad', 'reason': 'price invalid'},
]
policy_df = pd.DataFrame(policy_rows)
display(policy_df)

## Uso En Backtesting Y ML IA

In [ ]:
usage_df = pd.DataFrame([
    {'policy': 'good', 'backtesting': 'entra en core dataset', 'ml_train': 'entra en baseline', 'monitoring': 'normal'},
    {'policy': 'review', 'backtesting': 'solo en sensitivity runs', 'ml_train': 'entra con flag o menor peso', 'monitoring': 'seguir de cerca'},
    {'policy': 'bad', 'backtesting': 'fuera del core', 'ml_train': 'fuera del train principal', 'monitoring': 'set duro / anomalia'},
])
display(usage_df)

## Inspector Final De Casos

In [ ]:
schema_plus_vw = ensure_vw_refined(load_artifact('rescue_schema_plus_vw'))
focus = schema_plus_vw.copy()
focus['date_label'] = focus['year'].astype(int).astype(str) + '-' + focus['month'].astype(int).astype(str).str.zfill(2)
focus['case_label'] = focus['ticker'].astype(str) + ' | ' + focus['date_label'] + ' | ' + focus['vw_taxonomy'].astype(str) + ' | ratio=' + focus['vw_ratio_pct'].round(2).astype(str) + '%'

tax_dd = widgets.Dropdown(options=[(x, x) for x in sorted(focus['vw_taxonomy'].dropna().unique())], description='taxonomy', layout=widgets.Layout(width='320px'))
case_dd = widgets.Dropdown(description='file', layout=widgets.Layout(width='900px'))
out = widgets.Output()

def _refresh_cases(*_):
    sub = focus[focus['vw_taxonomy'].eq(tax_dd.value)].sort_values(['vw_ratio_pct','m.vw_outside_range_rows'], ascending=[False, False])
    case_dd.options = [(row['case_label'], int(idx)) for idx, row in sub.iterrows()]
    if len(case_dd.options):
        case_dd.value = case_dd.options[0][1]

def _render(*_):
    if case_dd.value is None:
        return
    row = focus.loc[int(case_dd.value)]
    with out:
        clear_output(wait=True)
        display(pd.DataFrame([row[[c for c in ['ticker','year','month','vw_taxonomy','quality_policy','m.rows_after_parse','m.vw_outside_range_rows','vw_ratio_pct','vw_per_active_day','m.active_days','m.max_gap_days','file'] if c in row.index]].to_dict()]))

tax_dd.observe(_refresh_cases, names='value')
case_dd.observe(_render, names='value')
_refresh_cases()
display(widgets.VBox([widgets.HBox([tax_dd]), case_dd, out]))
_render()